# 개별종목 조합K — XGBoost

`기본모델/03.XGBoost.ipynb`과 같은 `models.xgboost.build_xgboost_baseline`을 가져오고
조합K 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.xgboost import build_xgboost_baseline  # noqa: E402

MODEL_NAME = 'XGBoost'
MODEL_BUILDER = build_xgboost_baseline


In [2]:
# 2. 조합K의 피처 값만 지정합니다.
import json

COMBINATION = 'K'
FEATURE_COLUMNS = (
    'atr_ratio',
    'bb_bandwidth',
    'hv_regime',
    'five_day_return',
    'relative_ret_5_market',
    'sma_gap_5_20',
    'sma_gap_20_60',
    'rsi_14',
    'macd_hist_ratio',
    'bb_position',
    'hv_20',
    'vol_ratio_20',
    'obv_slope_20',
    'daily_return',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)
combination_report = report["combinations"].get(COMBINATION)
if combination_report is None:
    print("아직 실측 결과가 없습니다. 아래 공통 실행 명령으로 조합을 평가하세요.")
else:
    panel = combination_report["panel"]
    print("학습 기간:", panel["first_date"], "~", panel["last_date"])
    print("학습 행·종목:", panel["model_rows"], panel["stocks"])
    folds = pd.DataFrame(combination_report["outer_fold_results"])
    model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
    fold_columns = [
        "fold", "selected_class_weight", "train_dates", "valid_start", "valid_end",
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, fold_columns].round(4))
    metric_columns = [
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


조합K 피처: ('atr_ratio', 'bb_bandwidth', 'hv_regime', 'five_day_return', 'relative_ret_5_market', 'sma_gap_5_20', 'sma_gap_20_60', 'rsi_14', 'macd_hist_ratio', 'bb_position', 'hv_20', 'vol_ratio_20', 'obv_slope_20', 'daily_return')
학습 기간: 20110127 ~ 20240822
학습 행·종목: 159900 157


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,balanced_accuracy,mcc,pr_auc_macro_ovr,down_recall,core_harmonic_mean
0,1,balanced,750,20140217,20140514,0.4988,0.5012,-0.0024,0.3186,0.3630,0.0803,0.3823,0.0922,0.1877
1,2,balanced,980,20150123,20150421,0.3885,0.3978,-0.0093,0.3520,0.3630,0.0525,0.3811,0.2256,0.3046
2,3,balanced,1210,20151228,20160328,0.3703,0.3762,-0.0058,0.3704,0.3713,0.0601,0.3857,0.3698,0.3702
3,4,balanced,1439,20161202,20170228,0.4537,0.4617,-0.0081,0.3669,0.3814,0.0910,0.4136,0.1536,0.2623
4,5,balanced,1669,20171113,20180207,0.4210,0.3901,0.0309,0.3894,0.3991,0.1085,0.4069,0.2737,0.3489
5,6,balanced,1899,20181024,20190118,0.4107,0.3725,0.0382,0.4088,0.4140,0.1249,0.4311,0.4185,0.4126
6,7,balanced,2129,20190930,20191224,0.4638,0.4781,-0.0143,0.3683,0.3843,0.0999,0.4118,0.2370,0.3300
7,8,balanced,2359,20200902,20201130,0.3905,0.3476,0.0429,0.3875,0.3968,0.0946,0.4009,0.4901,0.4177
8,9,balanced,2589,20210806,20211105,0.3822,0.3914,-0.0093,0.3744,0.3916,0.0831,0.3894,0.2621,0.3295
9,10,balanced,2818,20220714,20221012,0.3709,0.3454,0.0255,0.3703,0.3772,0.0680,0.3880,0.2823,0.3356


,OOS 폴드 평균
accuracy,0.4130
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,0.0161
macro_f1,0.3755
balanced_accuracy,0.3867
mcc,0.0888
pr_auc_macro_ovr,0.3997
down_recall,0.3015
core_harmonic_mean,0.3421


재실행 명령: python scripts/run_stock_model_experiment.py
